# PEfE v2: evolve foraging agents in a world with movable bars

Clones the `v2-blocks` branch and evolves sparse recurrent-network agents in the v2 world
(`--world v2`): 40 heavy bars of length 300 that agents cannot pass through but can push.
Agents have 24 sensors (8 food, 8 agents, 8 bars). All behaviour is evolved from all-zero
genomes; the hand-coded controller is only an evaluation reference.

Evaluates the best genome on 20 held-out episodes, plus the *blind* check (food sensors zeroed)
and the *bar-blind* check (bar sensors zeroed), reports how far bars were moved and whether
several agents pushed together, and renders the team (orange = bar pushed by 2+ agents).
A generation takes about 4 s (about 25 minutes for 400). See `RESULTS_v2.md`.

In [ ]:
import os
BRANCH = os.environ.get("PEFE_BRANCH", "v2-blocks")
REPO_URL = os.environ.get("PEFE_REPO_URL", "https://github.com/PrimozRavbar/PEfE_exp.git")
SEED = int(os.environ.get("PEFE_SEED", 0))
GENERATIONS = int(os.environ.get("PEFE_GENERATIONS", 400))
RUN = f"runs/v2_seed{SEED}_gen{GENERATIONS}"

In [ ]:
# clone (fresh copy each time this cell runs)
import shutil
REPO_DIR = os.path.abspath("PEfE_exp")
shutil.rmtree(REPO_DIR, ignore_errors=True)
!git clone -q --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!pip -q install numpy matplotlib
if shutil.which("ffmpeg") is None:
    !apt-get -qq install -y ffmpeg > /dev/null
!git log --oneline -1

In [ ]:
# evolve in the v2 world (all other hyperparameters are the evolve.py defaults)
!python src/evolve.py --out {RUN} --world v2 --seed {SEED} --generations {GENERATIONS}

In [ ]:
# evaluate: 20 held-out episodes, blind (food sensors zeroed), bar-blind, bar statistics
!python src/evaluate.py --genome {RUN}/best.npz --json {RUN}/eval.json
import json
r = json.load(open(f"{RUN}/eval.json"))[0]
print(f"best validation {r['meta']['valid']:.2f}")
print(f"score {r['score']['mean']:.2f} ± {r['score']['sd']:.2f}   blind {r['blind']['mean']:.2f}   "
      f"bar-blind {r['bar_blind']['mean']:.2f}   connections {r['connections']}")
print(json.dumps(r['bars'], indent=1))

In [ ]:
# render the best clone team on the first held-out seed (bars drawn; start positions dashed)
!python src/render.py --genome {RUN}/best.npz --out {RUN}/best.mp4
from IPython.display import Image, Video, display
display(Image(f"{RUN}/best_trails.png"))
display(Video(f"{RUN}/best.mp4", embed=True, width=480))